LLM AS A JUGDE

Imagine you’re testing how good an AI is at answering questions.

Instead of a human grading every answer, you let another AI (a strong LLM) act like a judge.

The judge AI compares answers, scores them, or decides which one is better.

This saves time and makes evaluation scalable, but you have to be careful about bias (since it’s AI judging AI).

RAGAS Metrics
RAGAS = Retrieval-Augmented Generation Assessment.

It’s a set of metrics designed to check how well a RAG pipeline (where AI retrieves info + generates answers) is working.

Key things it measures:

Faithfulness: Is the answer actually based on the retrieved documents?

Answer relevance: Does the answer really address the question?

Context recall: Did the AI pull the right info from its sources?

Golden Datasets
A golden dataset is a carefully built set of examples with correct answers.

It’s like a “gold standard” reference you use to test AI.

Example: If you’re building a medical chatbot, your golden dataset might be 1,000 doctor‑verified Q&A pairs.

These datasets are critical because they give you a ground truth to measure against.

!pip install \
ragas==0.2.15 \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-core==0.3.59 \
langchain-openai==0.3.17 \
openai==1.76.0

In [1]:
import ragas
import langchain
import langchain_core
import langchain_community

print("ragas:", ragas.__version__)
print("langchain:", langchain.__version__)
print("langchain-core:", langchain_core.__version__)
print("langchain-community:", langchain_community.__version__)

ragas: 0.2.15
langchain: 0.3.25
langchain-core: 0.3.59
langchain-community: 0.3.24


In [2]:
from ragas import evaluate

print("✅ evaluate imported successfully")

✅ evaluate imported successfully


In [3]:
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

print("✅ All metrics imported successfully")

✅ All metrics imported successfully


In [1]:
from pypdf import PdfReader

# Load your project report PDF
reader = PdfReader("project_report_full.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:500])  # show first 500 characters


PROJECT REPORT
Project Name: Smart City Traffic Management System
Prepared By: Michael Johnson
Role: Senior Project Manager
Organization: UrbanTech Solutions Pvt Ltd
Project ID: SCTMS-2026-001
Project Start Date: January 5, 2026
Expected Completion Date: December 20, 2026
Project Budget: $2.5 Million
Location: Hyderabad, India
Project Objective
The Smart City Traffic Management System aims to reduce traffic congestion and improve road
safety through the use of artificial intelligence, real-time 


In [2]:
import spacy
from sentence_transformers import SentenceTransformer
import chromadb

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Semantic chunking function
def semantic_chunking(text, max_len=256):
    doc = nlp(text)
    chunks, current_chunk = [], ""
    for sent in doc.sents:
        if len(current_chunk) + len(sent.text) <= max_len:
            current_chunk += " " + sent.text
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sent.text
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks


In [3]:
# Assume you already extracted text from project_report_full.pdf
chunks = semantic_chunking(text, max_len=256)

print("Number of semantic chunks:", len(chunks))
print("First 3 chunks:", chunks[:3])


Number of semantic chunks: 13
First 3 chunks: ['', 'PROJECT REPORT\nProject Name: Smart City Traffic Management System\nPrepared By: Michael Johnson\nRole: Senior Project Manager\nOrganization: UrbanTech Solutions Pvt Ltd\nProject ID: SCTMS-2026-001\nProject Start Date: January 5, 2026\nExpected Completion Date: December 20, 2026\nProject Budget: $2.5 Million\nLocation:', 'Hyderabad, India\nProject Objective\nThe Smart City Traffic Management System aims to reduce traffic congestion and improve road\nsafety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics.']


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.get_or_create_collection("project_report")

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        embeddings=[model.encode([chunk])[0]],
        ids=[f"chunk_{i}"],
        metadatas=[{"section": "semantic"}]
    )


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
query = "What are the traffic management strategies in 2024?"
query_embedding = model.encode([query])[0]

results_internal = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Internal results:", results_internal["documents"])


Internal results: [['Conclusion\nThe Smart City Traffic Management System is progressing according to schedule and remains\nwithin the approved budget. The project team expects successful completion by December 2026.', 'PROJECT REPORT\nProject Name: Smart City Traffic Management System\nPrepared By: Michael Johnson\nRole: Senior Project Manager\nOrganization: UrbanTech Solutions Pvt Ltd\nProject ID: SCTMS-2026-001\nProject Start Date: January 5, 2026\nExpected Completion Date: December 20, 2026\nProject Budget: $2.5 Million\nLocation:', 'Hyderabad, India\nProject Objective\nThe Smart City Traffic Management System aims to reduce traffic congestion and improve road\nsafety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics.']]


In [6]:
import os
import boto3 
from dotenv import load_dotenv
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
load_dotenv("myenv.env")
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [7]:
import json 
prompt = f"""

Context:
{chr(10).join(results_internal["documents"][0])}

Question:
{query}


Answer based on the context. If partially available,
summarize whatever relevant information you can find.
"""

body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.2
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)
response = json.loads(response["body"].read())


print(response["output"]["message"]["content"][0]["text"])



Based on the provided context, there is no specific information about the traffic management strategies in 2024. The context focuses on the Smart City Traffic Management System project, which is set to start on January 5, 2026, and aims to be completed by December 20, 2026. The project's objective is to reduce traffic congestion and improve road safety using artificial intelligence, real-time traffic monitoring, and predictive analytics.

Since there is no direct information about traffic management strategies in 2024 within the provided context, it is not possible to provide details on those strategies based solely on the given information. For information on traffic management strategies in 2024, additional context or data would be required.


In [8]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall
)

import pandas as pd

In [9]:
questions = [
    "What are the traffic management strategies in 2024?",
    "What is the main objective of the project?",
    "What problem does the project solve?",
    "What technologies are used?",
    "What are the key findings?",
    "How is traffic congestion reduced?",
    "What datasets were used?",
    "What are the advantages of the proposed system?",
    "What challenges are discussed?",
    "What is the conclusion of the report?",
    "What recommendations are provided?",
    "How does the system improve efficiency?",
    "What methods are proposed?",
    "What algorithms are mentioned?",
    "What are the project goals?",
   
]

In [18]:
answers = []
contexts = []
ground_truths = []

for query in questions:

    # Retrieve relevant chunks
    query_embedding = model.encode([query])[0]

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_context = results["documents"][0]

    prompt = f"""
Context:
{chr(10).join(retrieved_context)}

Question:
{query}

Answer only from the given context.
"""

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ]
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body)
    )

    response_body = json.loads(response["body"].read())

    answer = response_body["output"]["message"]["content"][0]["text"]

    answers.append(answer)
    contexts.append(retrieved_context)

    # Placeholder ground truth
    # Replace these with actual answers if available.
    ground_truths.append(answer)


In [11]:
dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": ground_truths
})

dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 15
})

In [12]:
from langchain_aws import ChatBedrock
from ragas.llms import LangchainLLMWrapper

bedrock_llm = ChatBedrock(
    client=bedrock_runtime,
    model_id=MODEL_ID,
)

ragas_llm = LangchainLLMWrapper(bedrock_llm)

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

ragas_embeddings = LangchainEmbeddingsWrapper(embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall
)

In [15]:
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

In [16]:
print(result)

{'faithfulness': 0.9157, 'answer_relevancy': 0.5150, 'context_recall': 0.9000}


In [19]:
pip check

No broken requirements found.
Note: you may need to restart the kernel to use updated packages.
